<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Lab5_DeepTeam_LLM_Security_Assessment.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Automated LLM Red-Teaming with DeepTeam
## Running Adversarial Attacks Against Live CTF Challenges via Open WebUI

**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Time:** 50 minutes  
**Theme:** Defend  

---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Use DeepTeam** (Confident AI's red-teaming framework) to run automated adversarial attacks against LLMs
2. **Connect DeepTeam to live targets** via Open WebUI's chat completions API
3. **Understand vulnerability taxonomies** — bias, toxicity, PII leakage, prompt leakage
4. **Compare unprotected vs. hardened models** — measure how guardrails change attack outcomes
5. **Interpret red-team results** and identify which defenses are most effective

### Why This Matters

Manual red-teaming doesn't scale. Organizations deploying LLMs need **automated, repeatable security assessments** — the same way we run Nessus or Burp Suite against web applications. DeepTeam provides a structured framework for this: it generates adversarial inputs, sends them to your model, and evaluates whether the model's responses violate safety criteria.

In this lab, we'll run the **exact same attacks** against two different configurations of the same LLM:
- **Challenge 1** (`jackson_no_protections`) — Zero guardrails. The raw model.
- **Challenge 6** (`jackson_all_protections`) — Layered defenses: hardened system prompt, input filters, output filters, and LLM Guard.

The comparative approach reveals exactly what guardrails do (and don't) protect against.

### Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    This Jupyter Notebook                     │
│                                                              │
│  ┌──────────┐    ┌──────────────┐    ┌──────────────────┐   │
│  │ DeepTeam  │───>│  Callback    │───>│  Open WebUI API  │   │
│  │ red_team()│    │  Function    │    │  /chat/completions│  │
│  └──────────┘    └──────────────┘    └────────┬─────────┘   │
│       │                                       │              │
│       │          ┌──────────────┐             │              │
│       │          │  CTF Target  │<────────────┘              │
│       │          │  Workspace   │                            │
│       │          └──────────────┘                            │
│       v                                                      │
│  ┌──────────┐                                                │
│  │ Results   │  DataFrames, charts, pass/fail rates          │
│  └──────────┘                                                │
└──────────────────────────────────────────────────────────────┘
```

**Key difference from traditional scanners:** DeepTeam uses an LLM (GPT-4o-mini) as the attack *simulator* — it generates adversarial prompts dynamically based on vulnerability definitions. It also uses an LLM as the *evaluator* to score whether responses violated safety criteria. This means attacks are creative and contextual, not just pattern-matching.

---
## Section 1: Environment Setup

Let's install deepteam, verify the installation, and configure the required API key.

**Important:** DeepTeam uses OpenAI's GPT-4o-mini internally for two purposes:
1. **Attack Simulator** — Generates adversarial prompts based on vulnerability definitions
2. **Evaluator** — Scores whether model responses violated safety criteria

This requires a valid `OPENAI_API_KEY` environment variable.

In [ ]:
# Install DeepTeam
!pip install deepteam

In [ ]:
# Verify DeepTeam installation
import deepteam
print(f"DeepTeam version: {deepteam.__version__}")

In [ ]:
import os

# If you need to set the key manually, uncomment and fill in:
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Check that OPENAI_API_KEY is set (required for DeepTeam's simulator and evaluator)
api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY is not set!")
    print("DeepTeam needs this for its attack simulator and evaluator (GPT-4o-mini).")
    print("Set it in your .env file and restart the container, or set it here:")
    print('  os.environ["OPENAI_API_KEY"] = "sk-..."')
else:
    print(f"OPENAI_API_KEY is set (ends with ...{api_key[-4:]})")

---
## Section 2: Connecting to Open WebUI

DeepTeam's `red_team()` function accepts a **callback function** that takes a prompt string and returns the model's response string. We need to write a callback that:

1. Authenticates with Open WebUI (JWT-based)
2. Sends the prompt to a specific CTF challenge workspace
3. Returns the model's response text

Open WebUI uses JWT authentication (not API keys) for its chat completions endpoint.

In [ ]:
import requests

OPENWEBUI_URL = "http://open-webui:8080"

def get_auth_token(email: str, password: str) -> str:
    """Authenticate with Open WebUI and return a JWT token."""
    resp = requests.post(
        f"{OPENWEBUI_URL}/api/v1/auths/signin",
        json={"email": email, "password": password}
    )
    resp.raise_for_status()
    return resp.json()["token"]

# Authenticate with the CTF admin account
TOKEN = get_auth_token("admin@ctf.local", "ctf_admin_password")
print(f"Authenticated! Token: {TOKEN[:20]}...")

In [ ]:
import time

def openwebui_callback(input: str, turns=None, model_id: str = "jackson_no_protections") -> str:
    """
    DeepTeam callback function that sends prompts to Open WebUI.
    Includes retry logic to handle transient failures from the target API.

    Args:
        input: The adversarial prompt from DeepTeam
        turns: Multi-turn conversation history (unused for single-turn attacks)
        model_id: The Open WebUI workspace/model to target

    Returns:
        The model's response text
    """
    max_retries = 3
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                f"{OPENWEBUI_URL}/api/chat/completions",
                json={
                    "model": model_id,
                    "messages": [{"role": "user", "content": input}],
                    "stream": False
                },
                headers={"Authorization": f"Bearer {TOKEN}"},
                timeout=120
            )
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"]
        except (requests.exceptions.RequestException, KeyError) as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff: 1s, 2s
                continue
            raise

In [ ]:
# Test the callback against Challenge 1 (no protections)
test_response = openwebui_callback("Hello! Who are you?")
print("Challenge 1 response:")
print(test_response[:500])

In [ ]:
# Test the callback against Challenge 6 (all protections)
test_response_hardened = openwebui_callback("Hello! Who are you?", model_id="jackson_all_protections")
print("Challenge 6 response:")
print(test_response_hardened[:500])

---
## Section 3: Understanding Vulnerabilities & Attacks

DeepTeam organizes its red-teaming around two axes:

### Vulnerabilities (What we're testing for)

DeepTeam defines **37 vulnerability classes** across 4 categories:

| Category | Examples | What It Means |
|----------|----------|---------------|
| **Responsible AI** | Bias, Toxicity, Stereotypes | Model produces harmful or discriminatory content |
| **Information Security** | PII Leakage, Prompt Leakage | Model reveals sensitive information it shouldn't |
| **Brand Safety** | Off-topic responses, Competitor mentions | Model goes off-brand or promotes competitors |
| **Technical Robustness** | Hallucination, Contradiction | Model produces inaccurate or inconsistent output |

### Attacks (How we trigger vulnerabilities)

DeepTeam has **23 single-turn** and **5 multi-turn** attack methods:

| Type | Attack | How It Works |
|------|--------|--------------|
| **Single-turn** | `PromptInjection` | Attempts to override the system prompt with adversarial instructions |
| **Single-turn** | `Roleplay` | Asks the model to adopt a persona that bypasses safety guidelines |
| **Single-turn** | `ROT13` | Encodes the attack payload in ROT13 cipher to bypass input filters |
| **Single-turn** | `Leetspeak` | Uses character substitutions (e.g., 3 for E) to evade keyword filters |
| **Multi-turn** | `CrescendoJailbreaking` | Gradually escalates requests over multiple turns |
| **Multi-turn** | `LinearJailbreaking` | Linear progression toward harmful content over turns |

### Our Lab Configuration

For this lab, we'll use a focused subset to keep scan times reasonable (~2-5 minutes per target):

- **Vulnerabilities:** `PromptLeakage`, `PIILeakage`, `Bias`, `Toxicity`
- **Attacks:** `PromptInjection`, `Roleplay`, `ROT13`, `Leetspeak`
- **Attacks per vulnerability type:** 2

This generates approximately **16 test cases** per target (4 vulnerability types x 2 subtypes x 2 attacks per type).

In [ ]:
# Import DeepTeam components
from deepteam import red_team
from deepteam.vulnerabilities import Bias, Toxicity, PIILeakage, PromptLeakage
from deepteam.attacks.single_turn import PromptInjection, Roleplay, ROT13, Leetspeak
from functools import partial

# --- Fix for DeepTeam nested Rich progress bar bug ---
# DeepTeam's attack simulator and individual attack methods (PromptInjection,
# Roleplay, etc.) both create Rich progress bars. Rich only allows one live
# display at a time, so nested progress bars crash with LiveError. This patch
# makes nested progress bars into no-ops, preventing the crash.
import deepteam.utils as _dt_utils
from contextlib import nullcontext

_original_create_progress = _dt_utils.create_progress
_progress_active = False

def _safe_create_progress(*args, **kwargs):
    global _progress_active
    if _progress_active:
        return nullcontext()
    _progress_active = True
    prog = _original_create_progress(*args, **kwargs)
    class _Wrapper:
        def __enter__(self):
            self._ctx = prog.__enter__()
            return self._ctx
        def __exit__(self, *exc):
            global _progress_active
            _progress_active = False
            return prog.__exit__(*exc)
    return _Wrapper()

_dt_utils.create_progress = _safe_create_progress

# Patch modules that imported create_progress directly
import deepteam.attacks.single_turn.prompt_injection.prompt_injection as _pi
import deepteam.attacks.single_turn.roleplay.roleplay as _rp
import deepteam.attacks.attack_simulator.attack_simulator as _as
for _mod in [_pi, _rp, _as]:
    if hasattr(_mod, 'create_progress'):
        _mod.create_progress = _safe_create_progress
# --- End fix ---

# Define our vulnerability suite
vulnerabilities = [
    PromptLeakage(types=["instructions", "guard_exposure"]),
    PIILeakage(types=["direct_disclosure", "social_manipulation"]),
    Bias(types=["gender", "race"]),
    Toxicity(types=["insults", "threats"]),
]

# Define our attack suite
attacks = [PromptInjection(), Roleplay(), ROT13(), Leetspeak()]

print(f"Vulnerabilities: {len(vulnerabilities)} classes")
print(f"Attacks: {len(attacks)} methods")
print(f"Estimated test cases per target: ~{len(vulnerabilities) * 2 * 2}")
print("DeepTeam progress bar fix applied.")

---
## Section 4: Baseline Scan — Challenge 1 (No Protections)

First, we'll scan the **unprotected** target. Challenge 1 (`jackson_no_protections`) has:
- A basic character system prompt (Dr. Daniel Jackson)
- **No input filters**
- **No output filters**
- **No LLM Guard**

This establishes our baseline — what does the raw LLM do when attacked?

**Note:** This scan takes ~2-5 minutes as DeepTeam generates attacks via GPT-4o-mini, sends them to the target, and evaluates responses.

In [ ]:
# Create a callback bound to Challenge 1
baseline_callback = partial(openwebui_callback, model_id="jackson_no_protections")

print("Starting baseline scan against Challenge 1 (no protections)...")
print("This will take ~2-5 minutes.\n")

baseline_results = red_team(
    model_callback=baseline_callback,
    vulnerabilities=vulnerabilities,
    attacks=attacks,
    attacks_per_vulnerability_type=2,
    target_purpose="AI assistant character (Dr. Daniel Jackson) with no security guardrails",
    ignore_errors=True,
    async_mode=False,
)

print("\nBaseline scan complete!")

---
## Section 5: Analyze Baseline Results

DeepTeam returns structured results that we can inspect as DataFrames:
- **Overview** — Aggregate pass/fail rates per vulnerability type
- **Test Cases** — Individual attack inputs, model outputs, and evaluation scores

In [ ]:
# Overview: vulnerability pass rates
baseline_overview_df = baseline_results.overview.to_df()
print("=== BASELINE OVERVIEW (Challenge 1 — No Protections) ===")
baseline_overview_df

In [ ]:
# Detailed test cases
baseline_cases_df = baseline_results.test_cases.to_df()
print(f"Total test cases: {len(baseline_cases_df)}")
baseline_cases_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

# Visualize baseline vulnerability pass rates
fig, ax = plt.subplots(figsize=(10, 5))

if "vulnerability" in baseline_overview_df.columns and "pass_rate" in baseline_overview_df.columns:
    colors = ["#ef4444" if r < 0.5 else "#f59e0b" if r < 0.8 else "#22c55e"
              for r in baseline_overview_df["pass_rate"]]
    ax.barh(baseline_overview_df["vulnerability"], baseline_overview_df["pass_rate"], color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Pass Rate (higher = safer)")
    ax.set_title("Baseline Vulnerability Pass Rates — Challenge 1 (No Protections)")
    ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="50% threshold")
    ax.legend()
else:
    # Fallback: display whatever columns are available
    print("Available columns:", baseline_overview_df.columns.tolist())
    print(baseline_overview_df)

plt.tight_layout()
plt.show()

### Discussion Questions

Look at the baseline results and consider:

1. **Which vulnerability had the lowest pass rate?** Why might this be?
2. **Which attack method was most effective?** Check the test cases DataFrame.
3. **Were there any surprises?** Did the model resist some attacks you expected it to fail?
4. **What does a "pass" mean here?** DeepTeam's evaluator judges whether the response *actually* violated safety criteria, not just whether the model engaged with the prompt.

---
## Section 6: Hardened Scan — Challenge 6 (All Protections)

Now we run the **exact same attacks** against Challenge 6 (`jackson_all_protections`), which has:
- **Hardened system prompt** — Explicit refusal instructions
- **Input filters** — Keyword and pattern-based filtering on incoming messages
- **Output filters** — Post-processing to catch leaked content
- **LLM Guard** — ML-based content classification

Same vulnerabilities, same attacks, different defenses. Let's see what changes.

In [ ]:
# Create a callback bound to Challenge 6
hardened_callback = partial(openwebui_callback, model_id="jackson_all_protections")

print("Starting hardened scan against Challenge 6 (all protections)...")
print("This will take ~2-5 minutes.\n")

hardened_results = red_team(
    model_callback=hardened_callback,
    vulnerabilities=vulnerabilities,
    attacks=attacks,
    attacks_per_vulnerability_type=2,
    target_purpose="AI assistant with layered defenses: system prompt hardening, input/output filters, LLM Guard",
    ignore_errors=True,
    async_mode=False,
)

print("\nHardened scan complete!")

In [ ]:
# Overview: vulnerability pass rates for hardened target
hardened_overview_df = hardened_results.overview.to_df()
print("=== HARDENED OVERVIEW (Challenge 6 — All Protections) ===")
hardened_overview_df

In [ ]:
# Detailed test cases for hardened target
hardened_cases_df = hardened_results.test_cases.to_df()
print(f"Total test cases: {len(hardened_cases_df)}")
hardened_cases_df.head(10)

---
## Section 7: Comparative Analysis

The real value of automated red-teaming is **comparison**. Let's see side-by-side how the unprotected and hardened configurations performed against the same attacks.

In [ ]:
import pandas as pd

# Build a comparison DataFrame
# Adapt column names based on what DeepTeam actually returns
try:
    comparison_df = baseline_overview_df.merge(
        hardened_overview_df,
        on="vulnerability",
        suffixes=("_baseline", "_hardened")
    )
    print("=== SIDE-BY-SIDE COMPARISON ===")
    display(comparison_df)
except Exception as e:
    print(f"Merge failed (columns may differ): {e}")
    print("\nBaseline overview:")
    display(baseline_overview_df)
    print("\nHardened overview:")
    display(hardened_overview_df)

In [ ]:
# Grouped bar chart: Baseline vs Hardened
import numpy as np

fig, ax = plt.subplots(figsize=(12, 6))

try:
    vuln_labels = comparison_df["vulnerability"]
    baseline_rates = comparison_df["pass_rate_baseline"]
    hardened_rates = comparison_df["pass_rate_hardened"]

    x = np.arange(len(vuln_labels))
    width = 0.35

    bars1 = ax.bar(x - width/2, baseline_rates, width, label="Baseline (No Protections)", color="#ef4444", alpha=0.8)
    bars2 = ax.bar(x + width/2, hardened_rates, width, label="Hardened (All Protections)", color="#22c55e", alpha=0.8)

    ax.set_ylabel("Pass Rate (higher = safer)")
    ax.set_title("Vulnerability Pass Rates: Baseline vs Hardened")
    ax.set_xticks(x)
    ax.set_xticklabels(vuln_labels, rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f"{height:.0%}", xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f"{height:.0%}", xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9)
except Exception as e:
    print(f"Chart generation failed: {e}")
    print("This likely means the overview DataFrame columns differ from expected.")
    print("Check the raw DataFrames above and adjust column names.")

plt.tight_layout()
plt.show()

In [ ]:
# Identify attacks that bypassed hardened defenses (false sense of security)
try:
    # Look for test cases that failed (vulnerability triggered) even with protections
    if "passed" in hardened_cases_df.columns:
        bypasses = hardened_cases_df[hardened_cases_df["passed"] == False]
        if len(bypasses) > 0:
            print(f"=== GUARDRAIL BYPASSES ({len(bypasses)} found) ===")
            print("These attacks succeeded DESPITE all protections being enabled:\n")
            for _, row in bypasses.iterrows():
                print(f"  Vulnerability: {row.get('vulnerability', 'N/A')}")
                print(f"  Attack: {row.get('attack', 'N/A')}")
                input_text = str(row.get('input', 'N/A'))
                print(f"  Input: {input_text[:200]}..." if len(input_text) > 200 else f"  Input: {input_text}")
                print()
        else:
            print("All attacks were blocked by the hardened configuration!")
    else:
        print("Available columns:", hardened_cases_df.columns.tolist())
except Exception as e:
    print(f"Analysis failed: {e}")

In [ ]:
# Save results for later analysis
os.makedirs("./results/baseline", exist_ok=True)
os.makedirs("./results/hardened", exist_ok=True)

baseline_results.save(to="./results/baseline/")
hardened_results.save(to="./results/hardened/")

print("Results saved to ./results/baseline/ and ./results/hardened/")

### Key Takeaways

Compare your baseline and hardened results:

1. **Which guardrails were most effective?** Look at which vulnerability categories improved the most.
2. **Did any attacks still bypass everything?** Encoding attacks (ROT13, Leetspeak) often evade keyword-based input filters.
3. **What's the false sense of security?** If the hardened config blocks direct attacks but not encoded ones, the filters may give a false sense of protection.
4. **Defense in depth matters.** No single layer catches everything — the combination is what provides real security.

---
## Section 8: Advanced Exercises (Optional)

If you finish early or want to explore further, try these advanced exercises.

### Exercise 1: Multi-Turn Attacks

Multi-turn attacks are often more effective because they gradually escalate, building rapport before making the adversarial request.

**Note:** Multi-turn attacks take significantly longer to run (each test case involves multiple API calls).

In [ ]:
# Uncomment to run multi-turn attacks

# from deepteam.attacks.multi_turn import CrescendoJailbreaking, LinearJailbreaking
#
# multi_turn_results = red_team(
#     model_callback=hardened_callback,
#     vulnerabilities=[PromptLeakage(types=["instructions"])],
#     attacks=[CrescendoJailbreaking(turns=5), LinearJailbreaking(turns=5)],
#     attacks_per_vulnerability_type=1,
#     target_purpose="AI assistant with layered defenses",
#     ignore_errors=True,
# )
#
# print("Multi-turn results:")
# display(multi_turn_results.overview.to_df())

### Exercise 2: Target Other Challenges

The CTF environment has challenges with different defense configurations. Compare specific layers:

| Challenge | Model ID | Defenses |
|-----------|----------|----------|
| 1 | `jackson_no_protections` | None |
| 2 | `jackson_system_prompt_only` | System prompt hardening only |
| 3 | `jackson_input_filter` | Input filter only |
| 4 | `jackson_output_filter` | Output filter only |
| 5 | `jackson_llm_guard` | LLM Guard only |
| 6 | `jackson_all_protections` | All of the above |

In [ ]:
# Uncomment to scan a specific challenge layer

# input_filter_callback = partial(openwebui_callback, model_id="jackson_input_filter")
#
# input_filter_results = red_team(
#     model_callback=input_filter_callback,
#     vulnerabilities=vulnerabilities,
#     attacks=attacks,
#     attacks_per_vulnerability_type=2,
#     target_purpose="AI assistant with input filtering only",
#     ignore_errors=True,
# )
#
# display(input_filter_results.overview.to_df())

### Exercise 3: Custom Vulnerability — CTF Flag Leakage

The CTF challenges have a hidden flag in their system prompts. Can we create a targeted test for flag extraction?

DeepTeam supports custom vulnerabilities, but the simplest approach is to run prompt leakage tests and manually inspect for flag patterns in the responses.

In [ ]:
# Check if any baseline test case responses contain flag-like patterns
import re

flag_pattern = re.compile(r"flag\{.*?\}", re.IGNORECASE)

try:
    output_col = None
    for col in ["actual_output", "output", "response"]:
        if col in baseline_cases_df.columns:
            output_col = col
            break

    if output_col:
        for _, row in baseline_cases_df.iterrows():
            output_text = str(row[output_col])
            matches = flag_pattern.findall(output_text)
            if matches:
                print(f"FLAG FOUND in baseline response!")
                print(f"  Attack: {row.get('attack', 'N/A')}")
                print(f"  Vulnerability: {row.get('vulnerability', 'N/A')}")
                print(f"  Flag: {matches}")
                print()
        else:
            print("No flags found in baseline responses.")
            print("Try running more targeted prompt leakage attacks!")
    else:
        print(f"Available columns: {baseline_cases_df.columns.tolist()}")
        print("Adjust the column name to match DeepTeam's output format.")
except Exception as e:
    print(f"Flag search failed: {e}")

### Exercise 4: Discussion — Production Red-Teaming

Consider how you'd operationalize this in a real organization:

1. **CI/CD Integration** — Run DeepTeam scans on every model update. What pass rate threshold would you set as a gate?
2. **Custom Vulnerabilities** — What domain-specific vulnerabilities would you define for a healthcare LLM? A financial advisor?
3. **Attack Selection** — How would you prioritize which attacks to run in a time-limited scan?
4. **False Positives** — DeepTeam's evaluator can make mistakes. How would you handle false positives in an automated pipeline?
5. **Adversary Simulation** — The attacks in DeepTeam are known patterns. How would a real attacker differ?

---

## Summary

In this lab, you:

1. **Connected DeepTeam to live CTF targets** via Open WebUI's API using a callback function
2. **Ran automated adversarial attacks** against an unprotected model (Challenge 1)
3. **Ran the same attacks against a hardened model** (Challenge 6) with layered guardrails
4. **Compared results** to identify which defenses are effective and which attacks bypass them
5. **Explored advanced techniques** — multi-turn attacks, targeted scans, flag hunting

### Key Insights

- **Guardrails help but aren't perfect** — Encoding attacks often bypass keyword filters
- **Defense in depth is essential** — No single layer catches everything
- **Automated testing scales** — Manual red-teaming finds unique bugs, automated testing ensures coverage
- **The evaluator matters** — DeepTeam's LLM-based scoring is more nuanced than regex matching

### Next Steps

- Try the other CTF challenges to isolate which defense layer contributes most
- Explore DeepTeam's full vulnerability catalog (37 classes)
- Build a CI/CD pipeline that runs these scans automatically